In [ ]:
import requests
from datetime import datetime, timedelta

API_KEY = "b2a0d07341544104b699c5576a1bf7fd"
url = "https://newsapi.org/v2/everything"

# Query parameters
query = "finance AND (stock market OR investment)"
language = "en"
page_size = 100  # max per request

# Date range setup: last 30 days
end_date = datetime.today()
start_date = end_date - timedelta(days=30)

all_articles = []

# Split the 30-day range into 3-day chunks (adjust as needed)
delta = timedelta(days=3)
current_start = start_date

while current_start < end_date:
    current_end = min(current_start + delta, end_date)
    
    page = 1
    while True:
        params = {
            "q": query,
            "language": language,
            "pageSize": page_size,
            "page": page,
            "from": current_start.strftime("%Y-%m-%d"),
            "to": current_end.strftime("%Y-%m-%d"),
            "apiKey": API_KEY
        }

        response = requests.get(url, params=params)
        data = response.json()

        if "articles" in data and data["articles"]:
            all_articles.extend(data["articles"])
            if len(data["articles"]) < page_size:
                break  # No more pages for this date range
            page += 1
        else:
            break  # No articles for this page/date range

    current_start += delta

# Extract text
news_list = []
for article in all_articles:
    text = article.get("title", "") + " " + str(article.get("description", ""))
    news_list.append(text)

print(f"Total articles fetched: {len(news_list)}")
print(news_list[:10])

In [ ]:
from transformers import pipeline

# Load instruction-following LLM
summarizer = pipeline("text2text-generation", model="google/flan-t5-large")

# Example: take first 3 news articles
for article in news_list[:3]:
    prompt = f"""Summarize the article and analyze it for financial insights.
    Extract key points, mention if the sentiment is bullish, bearish, or neutral,
    and highlight any notable market events:\n{article}"""

    output = summarizer(prompt, max_length=150, do_sample=False)
    print("Original Article:\n", article)
    print("Summary + Analysis:\n", output[0]['generated_text'])
    print("---")

In [ ]:
from transformers import pipeline

summarizer = pipeline(
    "summarization",
    model="google/pegasus-xsum"
)

In [ ]:
summaries = []

for article in news_list[:5]:  # test on first 5 first
    summary = summarizer(
        article,
        max_length=int(len(article.split()) * 0.6),  # 60% of input length
        min_length=10,
        do_sample=False
    )
    
    summaries.append(summary[0]['summary_text'])

    print("Original:", article)
    print("Summary:", summary[0]['summary_text'])
    print("---")


In [ ]:
from transformers import pipeline

sentiment_model = pipeline("sentiment-analysis")

results = []

for summary in summaries:
    result = sentiment_model(summary)[0]
    results.append(result)

print(results[:5])

In [ ]:
def map_sentiment(label):
    if label == "POSITIVE":
        return "Bullish"
    elif label == "NEGATIVE":
        return "Bearish"
    else:
        return "Neutral"

mapped = [map_sentiment(r['label']) for r in results]
print(mapped[:5])